In [1]:
!pip install openai

In [3]:
from openai import AzureOpenAI
from google.colab import userdata

client=AzureOpenAI(
    api_key=userdata.get("AZURE_API_KEY"),
    azure_endpoint=userdata.get("AZURE_ENDPOINT"),
    api_version="2024-02-01"
)
print("❤️ Connected")

❤️ Connected


In [10]:
def bad_prompt_agent(ticket):
  response=client.chat.completions.create(
      model="gpt4nano-deploy",
      messages=[
          {
              "role":"system",
              "content":f"classify this: {ticket}"
          }
      ],
      max_completion_tokens=200
  )
  print("❌ Bad Prompt Response:")
  print(response.choices[0].message.content)
  print("-"*50)
bad_prompt_agent("My payment failed twice today!")

def good_prompt_agent(ticket):
  response=client.chat.completions.create(
      model="gpt4nano-deploy",
      messages=[
          {
              "role":"system",
              "content":"""You are an expert customer support triage agent.
              TASK: classify the customer ticket below.
              FORMAT: Always respond in EXACTLY this format, nothing else:
              Category: <issue category>
              Priority: <High/Medium/Low>
              Team: <team name>
              Reason: <one line reason>
              RULES:
              -Always follow exact format above
              -Never add extra information
              -Priority High=urgent financial/access issues
              -Priority Medium=general issues
              -Priority Low=information requests"""
          },
          {
              "role":"user",
              "content":f"Ticket: {ticket}"
          }
      ],
      max_completion_tokens=200
  )
  print("✅Good Prompt Response:")
  print(response.choices[0].message.content)
  print("-"*50)
good_prompt_agent("My payment failed twice today!")

❌ Bad Prompt Response:
**Category:** Billing/Payment Issue  
**Type:** Payment failed (transaction failed)
--------------------------------------------------
✅Good Prompt Response:
Category: Payment Failure
Priority: High
Team: Payments Support
Reason: Customer reports payment failing twice today, indicating an urgent transaction issue.
--------------------------------------------------


In [11]:
import json

def json_prompt_agent(ticket):
  response=client.chat.completions.create(
      model="gpt4nano-deploy",
      messages=[
          {
              "role":"system",
              "content":"""You are an expert triage agent.
              TASK: Analyze the customer ticket.
              FORMAT: Respond ONLY in valid JSON format like this:
              {
                "category":"issue category",
                "priority":"High/Medium/Low",
                "team":"team name",
                "reason":"one line reason",
                "estimated_resolution":"time estimate here",
                "sentiment":"Angry/Frustrated/Neutral/Happy"
              }
              RULES:
              -Return ONLY JSON, no other text
              -No markdown, no backticks
              -Always include all 6 fields"""
          },
          {
              "role":"user",
              "content":f"Ticket:{ticket}"
          }
      ],
      max_completion_tokens=200
  )
  raw = response.choices[0].message.content
  parsed = json.loads(raw)

  print("🎯 JSON Prompt Response:")
  print(f"  Category  : {parsed['category']}")
  print(f"  Priority  : {parsed['priority']}")
  print(f"  Team      : {parsed['team']}")
  print(f"  Reason    : {parsed['reason']}")
  print(f"  Resolution: {parsed['estimated_resolution']}")
  print(f"  Sentiment : {parsed['sentiment']}")
  print("-" * 50)

# Test it!
json_prompt_agent("My payment failed twice today!")

🎯 JSON Prompt Response:
  Category  : Payment failure
  Priority  : High
  Team      : Payments Support
  Reason    : Customer reports payment failing twice today, indicating urgent transaction/checkout issue.
  Resolution: 1-4 hours
  Sentiment : Frustrated
--------------------------------------------------


In [12]:
tickets = [
    "My payment failed twice today!",
    "I cannot login since morning!",
    "App is extremely slow and crashing!",
    "I was charged twice for same order!",
    "I need help with my account settings",
    "This is the worst app ever! Nothing works!"
]

print("🤖 AI Triage System - Full Analysis")
print("=" * 50)

for ticket in tickets:
    json_prompt_agent(ticket)

🤖 AI Triage System - Full Analysis
🎯 JSON Prompt Response:
  Category  : Payment failure
  Priority  : High
  Team      : Payments Support
  Reason    : Customer reports payment failures occurring twice on the same day.
  Resolution: 2-4 hours
  Sentiment : Frustrated
--------------------------------------------------
🎯 JSON Prompt Response:
  Category  : Authentication/Login issue
  Priority  : High
  Team      : Support/Identity & Access Management
  Reason    : User reports inability to log in since morning, indicating a likely account or authentication outage/problem.
  Resolution: 2-4 hours
  Sentiment : Frustrated
--------------------------------------------------
🎯 JSON Prompt Response:
  Category  : App performance and stability
  Priority  : High
  Team      : Mobile App Engineering
  Reason    : User reports severe slowness and crashes, indicating a critical performance/stability defect.
  Resolution: 4-8 hours (initial mitigation), up to 2-3 days for full fix
  Sentiment : A

In [14]:
# Complete Professional Triage System
# Day 3 - Prompt Engineering

def professional_triage_system(tickets):
    print("🏢 NIQ Professional Triage System")
    print("=" * 60)

    high_priority = []
    medium_priority = []
    low_priority = []
    angry_customers = []

    for ticket in tickets:
        response = client.chat.completions.create(
            model="gpt4nano-deploy",
            messages=[
                {
                    "role": "system",
                    "content": """You are an expert triage agent.
TASK: Analyze the customer ticket.
FORMAT: Respond ONLY in valid JSON:
{
    "category": "issue category",
    "priority": "High/Medium/Low",
    "team": "team name",
    "reason": "one line reason",
    "estimated_resolution": "time",
    "sentiment": "Angry/Frustrated/Neutral/Happy"
}
RULES: Return ONLY JSON, no other text"""
                },
                {
                    "role": "user",
                    "content": f"Ticket: {ticket}"
                }
            ],
            max_completion_tokens=200
        )

        raw = response.choices[0].message.content
        result = json.loads(raw)

        # Sort by priority
        if result['priority'] == 'High':
            high_priority.append((ticket, result))
        elif result['priority'] == 'Medium':
            medium_priority.append((ticket, result))
        else:
            low_priority.append((ticket, result))

        # Track angry customers
        if result['sentiment'] == 'Angry':
            angry_customers.append(ticket)

    # Print sorted results
    print("\n🔴 HIGH PRIORITY TICKETS:")
    for ticket, result in high_priority:
        print(f"  → {ticket[:40]}")
        print(f"     Team: {result['team']}")
        print(f"     Sentiment: {result['sentiment']}")

    print("\n🟡 MEDIUM PRIORITY TICKETS:")
    for ticket, result in medium_priority:
        print(f"  → {ticket[:40]}")
        print(f"     Team: {result['team']}")

    print("\n😡 ANGRY CUSTOMERS (Handle First!):")
    for ticket in angry_customers:
        print(f"  → {ticket[:40]}")

    print("\n📊 SUMMARY:")
    print(f"  High Priority   : {len(high_priority)}")
    print(f"  Medium Priority : {len(medium_priority)}")
    print(f"  Low Priority    : {len(low_priority)}")
    print(f"  Angry Customers : {len(angry_customers)}")

# Run it!
tickets = [
    "My payment failed twice today!",
    "I cannot login since morning!",
    "App is extremely slow and crashing!",
    "I was charged twice for same order!",
    "I need help with my account settings",
    "This is the worst app ever! Nothing works!"
]

professional_triage_system(tickets)

🏢 NIQ Professional Triage System

🔴 HIGH PRIORITY TICKETS:
  → My payment failed twice today!
     Team: Payments Support
     Sentiment: Frustrated
  → I cannot login since morning!
     Team: Support (Authentication)
     Sentiment: Frustrated
  → App is extremely slow and crashing!
     Team: Mobile/Web App Engineering
     Sentiment: Frustrated
  → I was charged twice for same order!
     Team: Payments Support
     Sentiment: Angry
  → This is the worst app ever! Nothing work
     Team: Mobile App Support
     Sentiment: Angry

🟡 MEDIUM PRIORITY TICKETS:
  → I need help with my account settings
     Team: Customer Support

😡 ANGRY CUSTOMERS (Handle First!):
  → I was charged twice for same order!
  → This is the worst app ever! Nothing work

📊 SUMMARY:
  High Priority   : 5
  Medium Priority : 1
  Low Priority    : 0
  Angry Customers : 2
